# 00 — Setup sanity check

Verifies:
1. `shared/` is importable
2. Mongo connection works and the expected collections are visible
3. Ollama is reachable at `LLM_BASE_URL` (default `http://localhost:11434`)
4. `data/` directories exist for outputs

Run this before any other notebook.

**Pre-req**: from `AI/` symlink the backend `.env`:
```bash
ln -sf ../.env .env
```

In [3]:
import sys, os
from pathlib import Path

# Make `shared/` importable when running notebooks from `notebooks/`.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))
print('AI_ROOT:', AI_ROOT)
print('Python :', sys.version.split()[0])

AI_ROOT: /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI
Python : 3.12.13


In [4]:
from shared.mongo_client import get_db, feedback_coll, agents_coll, count_by_category

db = get_db()
print('Mongo DB :', db.name)
print('feedback :', feedback_coll().estimated_document_count(), 'docs')
print('agents   :', agents_coll().estimated_document_count(), 'docs')

print('\nRule-category distribution:')
for cat, n in count_by_category().items():
    print(f'  {cat:20s} {n:>8,}')

Mongo DB : analyzed_agents
feedback : 186960 docs
agents   : 188615 docs

Rule-category distribution:
  config_feedback       120,817
  app_specific           48,965
  service_feedback        9,279
  others                  7,084
  spam                      782
  noise                      33


In [5]:
from shared.ollama_client import OllamaClient

# Reach Ollama with whatever model you have locally; falls back to qwen2.5:3b.
with OllamaClient(model='qwen2.5:3b', system_prompt='You are a test.') as cli:
    healthy = cli.health()
    print('Ollama healthy:', healthy)
    if not healthy:
        print('  → run `ollama serve` on the host; this notebook will not work without it.')

Ollama healthy: False
  → run `ollama serve` on the host; this notebook will not work without it.


In [6]:
for sub in ('splits', 'embeddings', 'results'):
    p = AI_ROOT / 'data' / sub
    p.mkdir(parents=True, exist_ok=True)
    print('ok ', p)

ok  /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits
ok  /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/embeddings
ok  /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results
